# Chapter 33
## M-Current PING and PINB
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter33.ipynb)

## About this chapter

PING (pyramidal-interneuron network gamma) normally repeats every gamma
cycle: E cells fire, drive the I cells, the I cells inhibit the E cells,
and as soon as that inhibition decays the E cells are ready to fire again.
The M-current is a slow, non-inactivating potassium current,

$$
I_M = \hat g_M\, w\,(v_K - v), \qquad
\dot w = \frac{w_\infty(v) - w}{\tau_w(v)},
$$

that builds up during a spike and only relaxes over tens of milliseconds.
When it is strong enough, an E cell that has just fired is left too
hyperpolarized to answer the very next gamma-timed volley of
disinhibition -- it skips a cycle. A population in which enough E cells
skip often enough produces a population rhythm at half (or a fraction of)
the gamma frequency: a beta rhythm built out of gamma-timescale spiking.

The examples below move from a single M-current population with gap
junctions (`M_CURRENT_BETA_WITH_GJ`), through five E-I PING networks that
vary recurrent excitation, drive, and connectivity to explore when and how
E cells skip (`M_CURRENT_PING_4`-`M_CURRENT_PING_8`), to PINB (PING with
I-cell "burst") networks in which the inhibitory population itself
organizes which E-cell assembly gets to fire and when, via two synaptic
pathways -- a fast I-I synapse and a slow I-E synapse -- driven by the
same I-cell spikes (`PINB_1`-`PINB_3`).

See [`chapter33.md`](chapter33.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math

import numpy as np
from numpy import exp, tanh
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit
from numba.typed import List

## Shared cell models and synaptic kernel (used by every example below)

Two cell models recur throughout: an RTM E cell, here augmented with the
M-current gate $w$, and a WB I cell (no M-current). Synapses use the usual
two-gate scheme -- a fast rise gate $q$ triggered by the presynaptic spike,
driving a slower gate $s$ (rise `tau_r`, decay `tau_d`) that is the actual
conductance -- with `tau_d_q_function` picking the release time constant so
that $s$ peaks at a prescribed `tau_peak` after the spike.

The network sims below integrate 40-250 coupled cells over tens of
thousands of time steps, so their per-timestep update loops are
`@njit`-compiled (numba); population initializers and the single bisection
searches for `tau_d_q` are plain NumPy, matching this repo's convention of
only accelerating the genuinely slow inner loops.

In [ ]:
# ------------------------------------------------------------- E cell (RTM + M-current)


def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


def w_inf(v):
    return 1. / (1 + exp(-(v + 35) / 10))


def tau_w(v):
    return 400. / (3.3 * exp((v + 35) / 20) + exp(-(v + 35) / 20))


# ------------------------------------------------------------- I cell (WB)


def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))
    beta_m = 4. * exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


def tau_h_i(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


def tau_n_i(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5


# --------------------------------------------------------- double-exp synapse


def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0., 0.
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2

In [ ]:
# ------------------------------------------------------- population splay init


def rtm_init_with_m_current_population(i_ext, phi_vec, g_m):
    '''vectorized RTM-with-M-current init over a population: each of
    len(i_ext) neurons is integrated (Heun/midpoint) independently until
    its 3rd spike, then (v,h,n,w) is interpolated at phase phi_vec[i]
    between the 2nd and 3rd spikes. Faithfully reproduces the matlab
    source's bug: m_tmp is computed from the pre-half-step v, not v_tmp.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    w = np.zeros(num)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 4))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, w_old, t_old = v, h, n, w, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
                  + g_l * (v_l - v) + g_m * w * (v_k - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)
        w_inc = (w_inf(v) - w) / tau_w(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc
        w_tmp = w + dt05_ * w_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)
        w_inc = (w_inf(v_tmp) - w_tmp) / tau_w(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        w = w + dt_ * w_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
            out[k, 3] = (w_old[k] * (t - thr[k]) + w[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    out[ind, 3] = w[ind]
    return out


def rtm_init_population(i_ext, phi_vec):
    '''same splay-state initializer as above, for the plain RTM E cell
    (no M-current) used by the PINB networks.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out

In [ ]:
# --------------------------------------------------- njit scalar gating kernels


@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit
def _w_inf_s(v):
    return 1. / (1 + math.exp(-(v + 35) / 10))


@njit
def _tau_w_s(v):
    return 400. / (3.3 * math.exp((v + 35) / 20) + math.exp(-(v + 35) / 20))


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.

## `M_CURRENT_BETA_WITH_GJ`: M-current beta timing with gap junctions

A single population of 200 M-current E cells, coupled only by sparse gap
junctions (electrical coupling proportional to $v_j-v_i$, no chemical
synapse). With the M-current strong enough, cells that just spiked skip
the next gamma-timed opportunity to fire together with the population,
and the gap junctions help synchronize which cells skip -- producing
population-level beta timing without any inhibitory population at all.

In [ ]:
def make_gap_junction_connectivity(num_e, g_hat_gap, p_gap, rng):
    g_gap = np.zeros((num_e, num_e))
    for i in range(num_e - 1):
        u = rng.random(num_e - 1 - i)
        connected = np.where(u < p_gap)[0] + i + 1
        g_gap[i, connected] = g_hat_gap / (num_e - 1) / p_gap
        g_gap[connected, i] = g_gap[i, connected]
    return g_gap


@njit
def _beta_gj_step_loop(m_steps, dt, dt05, num_e, g_m, i_ext_e, g_gap, g_gap_col_sum,
                        v_e, h_e, n_e, m_e, w):
    '''explicit-Heun per-timestep update for the M-current + gap-junction
    E population. Faithfully reproduces a quirk of the matlab source: the
    corrector stage does not recompute w_inc from the half-step state, so
    w itself is advanced by a plain Euler step (using the predictor's
    w_inc) even though v, h, n use a proper midpoint/Heun corrector.'''
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dhe = np.empty(num_e); dne = np.empty(num_e); dw = np.empty(num_e)
    ve_m = np.empty(num_e); he_m = np.empty(num_e); ne_m = np.empty(num_e)
    me_m = np.empty(num_e); we_m = np.empty(num_e)
    ve_old = np.empty(num_e)
    gap_term = np.empty(num_e)

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_gap[j, i] * v_e[j]
            gap_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + gap_term[j] - g_gap_col_sum[j] * v
                      + g_m * w[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            dw[j] = (_w_inf_s(v) - w[j]) / _tau_w_s(v)  # kept as-is through the corrector (see docstring)

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            we_m[j] = w[j] + dt05 * dw[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_gap[j, i] * ve_m[j]
            gap_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + gap_term[j] - g_gap_col_sum[j] * v
                      + g_m * we_m[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            # dw[j] intentionally not recomputed here

        for j in range(num_e):
            ve_old[j] = v_e[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            w[j] = w[j] + dt * dw[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))

    return e_times, e_indices


def simulate_m_current_beta_with_gj(g_hat_gap=0.2, p_gap=0.1, num_e=200, sigma_e=0.05,
                                     g_m=1.0, t_final=500., dt=0.01, seed=63806):
    '''M_CURRENT_BETA_WITH_GJ: a gap-junction-coupled population of
    M-current E cells (no synaptic coupling at all). Returns
    (t_e_spikes, i_e_spikes, num_e).

    MATLAB's rng('default'); rng(63806) can't be bit-reproduced by NumPy,
    so results are verified structurally, not against exact MATLAB spike
    times.'''
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    i_ext_e = 3.0 * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    g_gap = make_gap_junction_connectivity(num_e, g_hat_gap, p_gap, rng)
    g_gap_col_sum = g_gap.sum(axis=0)

    iv = rtm_init_with_m_current_population(i_ext_e, rng.random(num_e), g_m)
    v_e, h_e, n_e, w = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy(), iv[:, 3].copy()
    m_e = m_e_inf(v_e)

    e_times, e_indices = _beta_gj_step_loop(m_steps, dt, dt05, num_e, g_m, i_ext_e,
                                             g_gap, g_gap_col_sum, v_e, h_e, n_e, m_e, w)
    t_e_spikes = np.array(e_times) if len(e_times) else np.empty(0)
    i_e_spikes = np.array(e_indices) if len(e_indices) else np.empty(0, dtype=np.int64)
    return t_e_spikes, i_e_spikes, num_e


def plot_beta_gj_raster(t_e_spikes, i_e_spikes, num_e, t_final=500.):
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(t_e_spikes) > 0:
        ax.plot(t_e_spikes, i_e_spikes, '.r', markersize=2)
    ax.axis([0, t_final, 0, num_e + 1])
    ax.set_xlabel('$t$ [ms]')
    plt.tight_layout()
    plt.show()

In [ ]:
interact(lambda g_hat_gap=0.2: plot_beta_gj_raster(*simulate_m_current_beta_with_gj(g_hat_gap=g_hat_gap)),
         g_hat_gap=(0.0, 0.4, 0.02));

## `M_CURRENT_PING_4`-`M_CURRENT_PING_8`: period-skipping M-current PING

A shared E-I PING network (40 M-current E cells, 10 WB I cells) underlies
all five examples; they differ only in recurrent E-E excitation, I-cell
drive, and how the E population is wired up:

- **`M_CURRENT_PING_4`**: no recurrent E-E excitation, strong I drive.
- **`M_CURRENT_PING_5`**: adds weak all-to-all recurrent E-E excitation, weaker I drive.
- **`M_CURRENT_PING_6`**: recurrent E-E excitation confined to the second half
  of the E population (the "E_P" sub-assembly) only, run longer.
- **`M_CURRENT_PING_7`**: as `M_CURRENT_PING_6`, plus the first-half ("E_S")
  E-cells drive the I-cells only half as strongly.
- **`M_CURRENT_PING_8`**: as `M_CURRENT_PING_6`, plus the E_P sub-assembly is
  driven 20% harder (representing recurrent NMDA-receptor-mediated
  excitation).

Watch how many E cells skip successive gamma opportunities, and how that
depends on `g_m` (the M-current conductance shared by every E cell).

In [ ]:
def make_ping_m_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                             p_ee, p_ei, p_ie, p_ii, rng, ee_mode="random"):
    if ee_mode == "random":
        u_ee = rng.random((num_e, num_e))
        g_ee = g_hat_ee * (u_ee < p_ee) / (num_e * p_ee)
    else:
        # recurrent excitation among E_P-cells (the second half of the E
        # population) only, at a fixed (non-random) strength.
        g_ee = np.zeros((num_e, num_e))
        g_ee[num_e // 2:, num_e // 2:] = g_hat_ee / (num_e / 2)
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)
    return g_ee, g_ei, g_ie, g_ii


@njit
def _ping_m_step_loop(m_steps, dt, dt05, num_e, num_i,
                       v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                       tau_r_i, tau_d_i, tau_dq_i, g_m,
                       i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                       v_e, h_e, n_e, m_e, w, q_e, s_e,
                       v_i, h_i, n_i, m_i, q_i, s_i):
    '''explicit-Heun PING network stepper with an M-current on the E
    cells, shared by M_CURRENT_PING_4 through M_CURRENT_PING_8.'''
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e); dw = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e); we_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * w[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - w[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
            we_m[j] = w[j] + dt05 * dw[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v)
                      + g_m * we_m[j] * (-100 - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
            dw[j] = (_w_inf_s(v) - we_m[j]) / _tau_w_s(v)
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
            w[j] = w[j] + dt * dw[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

    return e_times, e_indices, i_times, i_indices


def simulate_m_current_ping(num_e=40, num_i=10, sigma_e=0.05, sigma_i=0.05,
                             i_ext_e_mean=3.0, i_ext_i_mean=0.7, g_m=0.5,
                             g_hat_ee=0.0, g_hat_ei=0.5, g_hat_ie=0.5, g_hat_ii=0.5,
                             p_ee=1.0, p_ei=1.0, p_ie=1.0, p_ii=1.0, ee_mode="random",
                             weaken_e_s_to_i=False, boost_e_p_drive=None,
                             v_rev_e=0., v_rev_i=-75.,
                             tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                             tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                             t_final=200., dt=0.01, seed=63806):
    '''shared, numba-accelerated M-current PING network stepper, reused
    by M_CURRENT_PING_4 through M_CURRENT_PING_8. Returns
    (t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes).

    MATLAB's rng('default'); rng(63806) can't be bit-reproduced by NumPy,
    so results are verified structurally, not against exact MATLAB spike
    times.'''
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    i_ext_e = i_ext_e_mean * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = i_ext_i_mean * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))
    if boost_e_p_drive is not None:
        # drive the E_P-cells harder (default 20%) to simulate effects of
        # recurrent NMDA-receptor-mediated excitation
        i_ext_e[num_e // 2:] *= boost_e_p_drive

    g_ee, g_ei, g_ie, g_ii = make_ping_m_connectivity(
        num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, p_ee, p_ei, p_ie, p_ii,
        rng, ee_mode=ee_mode)
    if weaken_e_s_to_i:
        # weaken the synapses from E_S-cells (the first half of the E
        # population) to I-cells
        g_ei[:num_e // 2, :] /= 2

    iv = rtm_init_with_m_current_population(i_ext_e, rng.random(num_e), g_m)
    v_e, h_e, n_e, w = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy(), iv[:, 3].copy()
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    e_times, e_indices, i_times, i_indices = _ping_m_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i, g_m,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, w, q_e, s_e,
        v_i, h_i, n_i, m_i, q_i, s_i)

    t_e_spikes = np.array(e_times) if len(e_times) else np.empty(0)
    i_e_spikes = np.array(e_indices) if len(e_indices) else np.empty(0, dtype=np.int64)
    t_i_spikes = np.array(i_times) if len(i_times) else np.empty(0)
    i_i_spikes = np.array(i_indices) if len(i_indices) else np.empty(0, dtype=np.int64)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes


def plot_m_current_ping_raster(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes,
                                num_e=40, num_i=10, t_final=200.):
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(t_i_spikes) > 0:
        ax.plot(t_i_spikes, i_i_spikes, '.b', markersize=6)
    if len(t_e_spikes) > 0:
        ax.plot(t_e_spikes, i_e_spikes + num_i, '.r', markersize=6)
    ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([num_i, num_e + num_i])
    ax.axis([0, t_final, 0, num_e + num_i + 1])
    ax.set_xlabel('$t$ [ms]')
    plt.tight_layout()
    plt.show()

In [ ]:
def simulate_m_current_ping_4(g_m=0.5):
    return simulate_m_current_ping(i_ext_i_mean=0.7, g_hat_ee=0.0, g_m=g_m, t_final=200.)

interact(lambda g_m=0.5: plot_m_current_ping_raster(*simulate_m_current_ping_4(g_m=g_m), t_final=200.),
         g_m=(0.0, 1.5, 0.05));

In [ ]:
def simulate_m_current_ping_5(g_hat_ee=0.35):
    return simulate_m_current_ping(i_ext_i_mean=0.3, g_hat_ee=g_hat_ee, t_final=200.)

interact(lambda g_hat_ee=0.35: plot_m_current_ping_raster(*simulate_m_current_ping_5(g_hat_ee=g_hat_ee), t_final=200.),
         g_hat_ee=(0.0, 0.6, 0.05));

In [ ]:
def simulate_m_current_ping_6(g_hat_ee=0.35):
    return simulate_m_current_ping(i_ext_i_mean=0.7, g_hat_ee=g_hat_ee, ee_mode="recurrent_half",
                                    t_final=500.)

interact(lambda g_hat_ee=0.35: plot_m_current_ping_raster(*simulate_m_current_ping_6(g_hat_ee=g_hat_ee), t_final=500.),
         g_hat_ee=(0.0, 0.6, 0.05));

In [ ]:
def simulate_m_current_ping_7(i_ext_i_mean=0.7):
    return simulate_m_current_ping(i_ext_i_mean=i_ext_i_mean, g_hat_ee=0.35, ee_mode="recurrent_half",
                                    weaken_e_s_to_i=True, t_final=500.)

interact(lambda i_ext_i_mean=0.7: plot_m_current_ping_raster(*simulate_m_current_ping_7(i_ext_i_mean=i_ext_i_mean), t_final=500.),
         i_ext_i_mean=(0.2, 1.2, 0.05));

In [ ]:
def simulate_m_current_ping_8(boost=1.20):
    return simulate_m_current_ping(i_ext_i_mean=0.7, g_hat_ee=0.35, ee_mode="recurrent_half",
                                    boost_e_p_drive=boost, t_final=500.)

interact(lambda boost=1.20: plot_m_current_ping_raster(*simulate_m_current_ping_8(boost=boost), t_final=500.),
         boost=(1.0, 1.6, 0.05));

## `PINB_1`-`PINB_3`: PING with I-cell "burst" (PINB)

PINB networks (200 RTM E cells, 50 WB I cells, no M-current) let the
inhibitory population itself organize which E-cell assembly fires and
when. Every I-cell spike drives *two* independent synaptic pathways: a
fast I-I synapse (`tau_d_ii`) that keeps the I population synchronized,
and a separate, typically slower, I-E synapse (`tau_d_ie`) that paces the
E cells. Because the two pathways gate independently even though both are
triggered by the same spikes, the E-inhibiting pulse can be made to
outlast the I-I pulse -- the I-cell "burst" that gives PINB its name.

- **`PINB_1`** builds the burst directly through synaptic kinetics: fast
  I-I (`tau_d_ii=9`) and slow I-E (`tau_d_ie=90`) at matched, modest
  strength (`g_hat_ie=0.25`).
- **`PINB_2`** instead uses a single (fast) synaptic timescale for both
  pathways, and gets the same qualitative effect from a very strong I-E
  weight (`g_hat_ie=10`).
- **`PINB_3`** is `PINB_2` with much weaker E-cell drive, showing how
  sparser E participation changes which assemblies the I-burst selects.

In [ ]:
def make_pinb_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii,
                            p_ee, p_ei, p_ie, p_ii, rng):
    u_ee = rng.random((num_e, num_e))
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ee = g_hat_ee * (u_ee < p_ee) / (num_e * p_ee) if p_ee > 0 else np.zeros((num_e, num_e))
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)
    return g_ee, g_ei, g_ie, g_ii


@njit
def _pinb_step_loop(m_steps, dt, dt05, num_e, num_i,
                     v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                     tau_r_i, tau_d_ii, tau_dq_ii, tau_d_ie, tau_dq_ie,
                     i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                     v_e, h_e, n_e, m_e, q_e, s_e,
                     v_i, h_i, n_i, m_i, q_ii, s_ii, q_ie, s_ie):
    '''explicit-Heun PINB network stepper: plain RTM/WB cells (no
    M-current), but the I population drives two independent synapse pools
    -- (q_ii, s_ii) onto other I-cells and (q_ie, s_ie) onto E-cells --
    from the same v_i. Shared by PINB_1 through PINB_3. Also returns the
    E-cell-averaged LFP-like trace.'''
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqii = np.empty(num_i); dsii = np.empty(num_i)
    dqie = np.empty(num_i); dsie = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i)
    qii_m = np.empty(num_i); sii_m = np.empty(num_i)
    qie_m = np.empty(num_i); sie_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    lfp = np.empty(m_steps + 1)
    acc = 0.0
    for j in range(num_e):
        acc += v_e[j]
    lfp[0] = acc / num_e

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_ie[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_ii[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqii[j] = (1 + th) / 2 * (1 - q_ii[j]) / 0.1 - q_ii[j] / tau_dq_ii
            dqie[j] = (1 + th) / 2 * (1 - q_ie[j]) / 0.1 - q_ie[j] / tau_dq_ie
            dsii[j] = q_ii[j] * (1 - s_ii[j]) / tau_r_i - s_ii[j] / tau_d_ii
            dsie[j] = q_ie[j] * (1 - s_ie[j]) / tau_r_i - s_ie[j] / tau_d_ie

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qii_m[j] = q_ii[j] + dt05 * dqii[j]
            sii_m[j] = s_ii[j] + dt05 * dsii[j]
            qie_m[j] = q_ie[j] + dt05 * dqie[j]
            sie_m[j] = s_ie[j] + dt05 * dsie[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * sie_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * sii_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqii[j] = (1 + th) / 2 * (1 - qii_m[j]) / 0.1 - qii_m[j] / tau_dq_ii
            dsii[j] = qii_m[j] * (1 - sii_m[j]) / tau_r_i - sii_m[j] / tau_d_ii
            dqie[j] = (1 + th) / 2 * (1 - qie_m[j]) / 0.1 - qie_m[j] / tau_dq_ie
            dsie[j] = qie_m[j] * (1 - sie_m[j]) / tau_r_i - sie_m[j] / tau_d_ie

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_ii[j] = q_ii[j] + dt * dqii[j]
            s_ii[j] = s_ii[j] + dt * dsii[j]
            q_ie[j] = q_ie[j] + dt * dqie[j]
            s_ie[j] = s_ie[j] + dt * dsie[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

        acc = 0.0
        for j in range(num_e):
            acc += v_e[j]
        lfp[k] = acc / num_e

    return e_times, e_indices, i_times, i_indices, lfp


def simulate_pinb(num_e=200, num_i=50, sigma_e=0.05, i_ext_e_mean=1.4,
                   sigma_i=0.0, i_ext_i_mean=0.0,
                   g_hat_ee=0.0, g_hat_ei=0.25, g_hat_ie=0.25, g_hat_ii=0.25,
                   p_ee=0.5, p_ei=0.5, p_ie=0.5, p_ii=0.5,
                   tau_d_ii=9.0, tau_d_ie=9.0,
                   v_rev_e=0., v_rev_i=-75.,
                   tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                   tau_r_i=0.5, tau_peak_i=0.5,
                   t_final=400., dt=0.01, seed=63806):
    '''shared, numba-accelerated PINB network stepper, reused by PINB_1
    through PINB_3. Returns (t_e_spikes, i_e_spikes, t_i_spikes,
    i_i_spikes, lfp, m_steps).

    MATLAB's rng('default'); rng(63806) can't be bit-reproduced by NumPy,
    so results are verified structurally, not against exact MATLAB spike
    times.'''
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_ii = tau_d_q_function(tau_d_ii, tau_r_i, tau_peak_i)
    tau_dq_ie = tau_d_q_function(tau_d_ie, tau_r_i, tau_peak_i)

    i_ext_e = i_ext_e_mean * np.ones(num_e) * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = i_ext_i_mean * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))

    g_ee, g_ei, g_ie, g_ii = make_pinb_connectivity(
        num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, p_ee, p_ei, p_ie, p_ii, rng)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_ii, s_ii = np.zeros(num_i), np.zeros(num_i)
    q_ie, s_ie = np.zeros(num_i), np.zeros(num_i)

    e_times, e_indices, i_times, i_indices, lfp = _pinb_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_ii, tau_dq_ii, tau_d_ie, tau_dq_ie,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e,
        v_i, h_i, n_i, m_i, q_ii, s_ii, q_ie, s_ie)

    t_e_spikes = np.array(e_times) if len(e_times) else np.empty(0)
    i_e_spikes = np.array(e_indices) if len(e_indices) else np.empty(0, dtype=np.int64)
    t_i_spikes = np.array(i_times) if len(i_times) else np.empty(0)
    i_i_spikes = np.array(i_indices) if len(i_indices) else np.empty(0, dtype=np.int64)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, np.array(lfp), m_steps


def plot_pinb(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, lfp, m_steps,
              num_e=200, num_i=50, t_final=400., dt=0.01):
    fig, axes = plt.subplots(2, 1, figsize=(8, 6))

    ax = axes[0]
    if len(t_i_spikes) > 0:
        ax.plot(t_i_spikes, i_i_spikes, '.b', markersize=2)
    if len(t_e_spikes) > 0:
        ax.plot(t_e_spikes, i_e_spikes + num_i, '.r', markersize=2)
    ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([num_i, num_e + num_i])
    ax.axis([0, t_final, 0, num_e + num_i + 1])

    axes[1].plot(np.arange(m_steps + 1) * dt, lfp, '-k', linewidth=2)
    axes[1].set_xlabel('$t$ [ms]')
    axes[1].set_ylabel('mean($v$), E-cells')
    axes[1].axis([0, t_final, -100, 50])

    plt.tight_layout()
    plt.show()

In [ ]:
def simulate_pinb_1(tau_d_ie=90.0):
    return simulate_pinb(i_ext_e_mean=1.4, g_hat_ie=0.25, tau_d_ii=9.0, tau_d_ie=tau_d_ie)

interact(lambda tau_d_ie=90.0: plot_pinb(*simulate_pinb_1(tau_d_ie=tau_d_ie)),
         tau_d_ie=(9.0, 150.0, 5.0));

In [ ]:
def simulate_pinb_2(g_hat_ie=10.0):
    return simulate_pinb(i_ext_e_mean=1.4, g_hat_ie=g_hat_ie, tau_d_ii=9.0, tau_d_ie=9.0)

interact(lambda g_hat_ie=10.0: plot_pinb(*simulate_pinb_2(g_hat_ie=g_hat_ie)),
         g_hat_ie=(0.0, 15.0, 0.5));

In [ ]:
def simulate_pinb_3(i_ext_e_mean=0.4):
    return simulate_pinb(i_ext_e_mean=i_ext_e_mean, g_hat_ie=10.0, tau_d_ii=9.0, tau_d_ie=9.0)

interact(lambda i_ext_e_mean=0.4: plot_pinb(*simulate_pinb_3(i_ext_e_mean=i_ext_e_mean)),
         i_ext_e_mean=(0.2, 2.0, 0.1));